In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))



In [2]:
from src.io.pitch_io import load_preprocessed_pitch, load_pitch_file
from src.io.annotation_io import load_annotations
import settings as S

This block defines the parameters that control how pitch contours are converted into training data.

Key parameters:

- `WINDOW_SIZE`: number of samples per training window
- `STRIDE`: distance between consecutive windows
- `MIN_RUN_LEN`: minimum number of valid samples required for a contour fragment
- `NORMALIZE_PER_WINDOW`: whether each window is z-score normalized

These parameters determine how much local context the model sees and how many training examples are generated.

In [3]:
SEED = 42

WINDOW_SIZE = 32
STRIDE = 8
NORMALIZE_PER_WINDOW = True

MIN_RUN_LEN = 32

START_COL = "start_time_sec"
END_COL = "end_time_sec"

In [4]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Each training window can optionally be normalized using z-score normalization.

This transforms the window so that:

- mean = 0
- standard deviation = 1

Normalization helps the model focus on the **shape of the pitch trajectory** rather than absolute pitch height.

Since the pitch is already expressed in cents relative to the tonic, normalization mainly removes local offset differences.

In [5]:
def zscore_normalize(x, eps=1e-8):
    mean = x.mean()
    std = x.std()

    if std < eps:
        return x - mean

    return (x - mean) / (std + eps)

Long pitch contours are split into fixed-length windows.

Given a contour:

- windows of length `WINDOW_SIZE` are extracted
- consecutive windows are separated by `STRIDE` samples

This produces overlapping windows and increases the number of training examples.

For example, a contour of 1000 samples can produce dozens of training windows.

In [ ]:
def slice_windows(contour, window_size=256, stride=64):
    windows = []
    n = len(contour)

    if n < window_size:
        return windows

    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        windows.append(contour[start:end])

    return windows


def build_window_corpus(clean_contours,
                        window_size=32,
                        stride=8,
                        normalize_per_window=True):
    all_windows = []

    for contour in clean_contours:
        windows = slice_windows(contour, window_size=window_size, stride=stride)

        for w in windows:
            w = np.asarray(w, dtype=np.float32)

            if normalize_per_window:
                w = zscore_normalize(w)

            all_windows.append(w.astype(np.float32))

    if len(all_windows) == 0:
        raise ValueError("No windows were created. Check MIN_RUN_LEN and WINDOW_SIZE.")

    return np.stack(all_windows, axis=0)

A boolean mask is constructed to identify pitch samples that are considered reliable.

This mask uses information already computed during preprocessing, such as:

- `too_long_to_interp`
- `is_outlier`
- `valid_for_pchip`

Only samples that pass these checks are considered valid.

Using these existing flags avoids repeating preprocessing steps and ensures consistency with the pitch cleaning pipeline.

In [7]:
def build_valid_sample_mask(df_pitch, df_pitch_aux, pitch_col):
    pitch_vals = df_pitch[pitch_col].to_numpy()
    valid_mask = np.isfinite(pitch_vals)

    if "too_long_to_interp" in df_pitch_aux.columns:
        valid_mask &= ~df_pitch_aux["too_long_to_interp"].to_numpy()

    if "is_outlier" in df_pitch_aux.columns:
        valid_mask &= ~df_pitch_aux["is_outlier"].to_numpy()

    if "valid_for_pchip" in df_pitch_aux.columns:
        valid_mask &= df_pitch_aux["valid_for_pchip"].to_numpy()

    return valid_mask

Within each segment, the code identifies contiguous sequences of valid samples.

Instead of removing invalid samples and compressing the contour, the algorithm detects **runs of consecutive valid points**.

Each run represents a temporally continuous portion of the pitch trajectory.

This prevents two valid fragments separated by invalid samples from being artificially merged into a single contour.

In [8]:

def find_true_runs(mask):
    """
    Returns a list of (start_idx, end_idx) pairs for contiguous True runs.
    end_idx is exclusive.
    """
    runs = []
    n = len(mask)

    i = 0
    while i < n:
        if mask[i]:
            start = i
            while i < n and mask[i]:
                i += 1
            end = i
            runs.append((start, end))
        else:
            i += 1

    return runs


Pitch contours are extracted for each annotated svara segment.

For every annotation:

1. The corresponding time interval is selected from the pitch dataframe.
2. The validity mask is applied.
3. Continuous valid runs are detected.
4. Runs shorter than `MIN_RUN_LEN` are discarded.

Each remaining run is stored as an independent pitch contour.

This produces a collection of clean pitch fragments that can be used for training.

In [9]:
def extract_segment_contours(df_pitch,
                             df_pitch_aux,
                             df_svaras,
                             pitch_col,
                             min_run_len=32):
    time_vals = df_pitch[S.TIME_COL].to_numpy()
    pitch_vals = df_pitch[pitch_col].to_numpy()

    valid_mask_global = build_valid_sample_mask(df_pitch, df_pitch_aux, pitch_col)

    starts = df_svaras[START_COL].to_numpy()
    ends = df_svaras[END_COL].to_numpy()

    clean_contours = []

    for start_sec, end_sec in zip(starts, ends):
        seg_mask = (time_vals >= start_sec) & (time_vals <= end_sec)

        seg_pitch = pitch_vals[seg_mask]
        seg_valid = valid_mask_global[seg_mask]

        runs = find_true_runs(seg_valid)

        for start_idx, end_idx in runs:
            contour = seg_pitch[start_idx:end_idx]

            if len(contour) < min_run_len:
                continue

            contour = np.asarray(contour, dtype=np.float32)

            if not np.all(np.isfinite(contour)):
                continue

            clean_contours.append(contour)

    if len(clean_contours) == 0:
        raise ValueError("No valid continuous contours were extracted.")

    return clean_contours

Two datasets are used:

- the **preprocessed pitch trajectory** of the recording
- the **svara annotation file**

The pitch is loaded with the option `convert_to_cents=True`, which converts frequencies from Hz to cents relative to the tonic.

Working in cents makes pitch trajectories comparable across recordings with different tonic frequencies.

In [10]:
set_seed(SEED)

recording_id = S.CURRENT_PIECE
tonic_hz = S.SARASUDA_TONICS[recording_id]
aux_pitch_path = S.DATA_INTERIM / recording_id / "pitch" / f"{recording_id}_pitch_preprocessed_debug.parquet"

annotation_path = (
    S.DATA_CORPUS
    / recording_id
    / "raw"
    / f"{recording_id}_ann_svara.tsv"
)

df_pitch = load_preprocessed_pitch(
    recording_id=recording_id,
    root_dir=S.DATA_INTERIM,
    tonic_hz=tonic_hz,
    convert_to_cents=True,
)

df_pitch_aux = load_pitch_file(
    file_path=aux_pitch_path
)

df_svaras = load_annotations(
    file_path=annotation_path,
    annotation_type="svara",
    engine="polars",
)

Using the previously defined functions, continuous pitch contours are extracted from the annotated segments.

Each contour corresponds to a continuous region of valid pitch samples within a svara segment.

The result is a list of 1D NumPy arrays, where each array represents a clean pitch trajectory fragment.

In [11]:
clean_contours = extract_segment_contours(
    df_pitch,
    df_pitch_aux,
    df_svaras=df_svaras,
    pitch_col=S.PITCH_COL_CENTS,
    min_run_len=MIN_RUN_LEN,
)

print("Number of clean continuous contours:", len(clean_contours))
print("First contour length:", len(clean_contours[0]))


Number of clean continuous contours: 388
First contour length: 58


The extracted contours are finally converted into the training corpus.

Each contour is split into overlapping windows using the parameters:

- `WINDOW_SIZE`
- `STRIDE`

The result is a matrix of shape: [N_windows, WINDOW_SIZE] where each row corresponds to one training example.

In [12]:

windows = build_window_corpus(
    clean_contours=clean_contours,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    normalize_per_window=NORMALIZE_PER_WINDOW,
)

print("Window corpus shape:", windows.shape)
print("Example window shape:", windows[0].shape)

Window corpus shape: (791, 32)
Example window shape: (32,)
